# EXP-007 - image-to-video on free Kaggle with Wan2.2-TI2V-5B

Conditions a video on **one still image**. Nothing here touches Supabase, the
worker, or any Cretona secret.

**Before Run All:**

1. Settings -> Accelerator -> **GPU T4** (one is enough)
2. Settings -> Internet -> **On**
3. Add Input -> upload one still image as a private dataset

Cells are ordered cheapest-first. Cells 1-3 cost seconds and download no
weights; they exist so a doomed run dies before 34 GB moves.

> Checkpoints are **apache-2.0**. Unlike EXP-006's MMAudio weights, there is no
> non-commercial condition attached.

## Cell 1 - the free checks: card, dtype, disk

Three ways this can be over before it starts, none of which cost a download.

The bfloat16 line matters more than it looks. The model card's own example runs
in bf16, and **Turing has no bf16**. Everything below runs the transformer in
float16 instead, which is what `kaggle_worker.py` already does for Wan2.1 -
but that is evidence about a different model, and kill criterion 5 exists
because of it.

In [ ]:
import platform, shutil, torch

print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('cuda   :', torch.version.cuda, '| available:', torch.cuda.is_available())

assert torch.cuda.is_available(), 'No GPU. Settings -> Accelerator -> GPU T4.'
prop = torch.cuda.get_device_properties(0)
vram_gb = prop.total_memory / 1e9
print('gpu    :', prop.name, f'{vram_gb:.2f} GB', f'| sm_{prop.major}{prop.minor}')
print('bf16   :', torch.cuda.is_bf16_supported(), '(False on T4/Turing, as expected)')

# Hugging Face caches under $HOME, not /kaggle/working. Check the right disk.
from pathlib import Path
cache = Path.home() / '.cache'
cache.mkdir(exist_ok=True)
free_gb = shutil.disk_usage(cache).free / 1e9
print()
print(f'free at {cache}: {free_gb:.1f} GB  (need ~40 for a 34.2 GB download)')

ok = free_gb >= 40
print()
print('DISK OK:', ok)
if not ok:
    print('KILL CRITERION 1 MET on disk. Record it in README.md and stop.')

## Cell 2 - install the *pinned* versions, and prove torch survived

These are the exact pins in `backend/requirements-kaggle.txt`. Installing
anything else here would measure a stack the worker does not run.

If pip drags in a different torch, **stop**. That is kill criterion 1 arriving
late, and it is the one failure in this notebook that can damage the working
video path rather than merely end the experiment.

In [ ]:
import torch
before = torch.__version__

!python -m pip install -q "diffusers==0.36.0" "transformers==4.57.3" "accelerate==1.12.0" "ftfy==6.3.1" "sentencepiece==0.2.1" "imageio==2.37.0" "imageio-ffmpeg==0.6.0" 2>&1 | tail -5

import importlib, torch
importlib.reload(torch)
print()
print('torch before:', before)
print('torch after :', torch.__version__)
assert torch.__version__ == before, 'pip changed torch. STOP - restart the session and do not proceed.'
print('torch unchanged.')

## Cell 3 - does the pinned diffusers actually know this model?

The model card says image-to-video needs diffusers from source. That note is
from August 2025 and is stale: the `expand_timesteps` path is in the released
`v0.35.0` and `v0.36.0`. This cell verifies that on the machine rather than
trusting the reading.

It also pulls three small JSON files - a few kilobytes, no weights - to confirm
what the repository claims about itself:

- `expand_timesteps: true`, so the conditioning tensor is one frame;
- `image_dim: null`, so **no CLIP image encoder is needed** and none is loaded;
- `boundary_ratio: null` and `transformer_2: null`, so this is a single
  transformer, not the two-expert MoE the 14B models use.

In [ ]:
import inspect, json
from huggingface_hub import hf_hub_download
import diffusers
from diffusers import WanImageToVideoPipeline

MODEL_ID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
MODEL_REVISION = 'b8fff7315c768468a5333511427288870b2e9635'

print('diffusers:', diffusers.__version__)
params = inspect.signature(WanImageToVideoPipeline.__init__).parameters
print('expand_timesteps in __init__:', 'expand_timesteps' in params)
assert 'expand_timesteps' in params, 'KILL CRITERION 1: this diffusers cannot run Wan2.2 I2V.'

def peek(name):
    path = hf_hub_download(MODEL_ID, name, revision=MODEL_REVISION)
    return json.load(open(path))

index = peek('model_index.json')
transformer_cfg = peek('transformer/config.json')
vae_cfg = peek('vae/config.json')

print()
print('expand_timesteps :', index.get('expand_timesteps'), '(want True)')
print('boundary_ratio   :', index.get('boundary_ratio'), '(want None - single transformer)')
print('transformer_2    :', index.get('transformer_2'), '(want [None, None])')
print('image_dim        :', transformer_cfg.get('image_dim'), '(want None - no CLIP encoder)')
print('vae spatial/temporal:', vae_cfg.get('scale_factor_spatial'), '/', vae_cfg.get('scale_factor_temporal'))

## Cell 4 - the shot: image in, geometry out

Resolves whatever image you attached without assuming a dataset mount path,
then prints the sequence length this configuration will actually cost.

That arithmetic is the reason this experiment exists. The model card's 24 GB
figure is for **1280x704 at 121 frames = 27,280 tokens**. The numbers printed
below are what a Cretona job would really ask for.

In [ ]:
from pathlib import Path
from PIL import Image

SUFFIXES = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
found = sorted(p for p in Path('/kaggle/input').rglob('*') if p.suffix.lower() in SUFFIXES)
assert found, 'No image under /kaggle/input. Attach a dataset containing one still image.'
for p in found:
    print(f'{p.stat().st_size:>10,} B  {p}')

REFERENCE = found[0]
image = Image.open(REFERENCE).convert('RGB')
print()
print('using:', REFERENCE, image.size)

# Cretona's own 16:9 geometry. check_inputs() requires both divisible by 16.
WIDTH, HEIGHT = 832, 480
FRAMES = 97          # 4 seconds at the model's native 24 fps; must be 4n+1
FPS = 24
STEPS = 25           # same step count the worker runs today
GUIDANCE = 5.0       # the model card's value

assert WIDTH % 16 == 0 and HEIGHT % 16 == 0
assert (FRAMES - 1) % 4 == 0

t_lat = (FRAMES - 1) // 4 + 1
h_lat, w_lat = HEIGHT // 16, WIDTH // 16
tokens = t_lat * (h_lat // 2) * (w_lat // 2)
print()
print(f'{WIDTH}x{HEIGHT}, {FRAMES} frames @ {FPS} fps = {FRAMES / FPS:.2f} s')
print(f'latent grid  : {t_lat} x {h_lat} x {w_lat}')
print(f'tokens       : {tokens:,}')
print(f'  vs 720p reference config (24 GB claim) : 27,280')
print(f'  vs Wan2.1-1.3B at 832x480x65 today     : 26,520')

image = image.resize((WIDTH, HEIGHT), Image.LANCZOS)

## Cell 5 - phase 1: encode the prompt on **CPU**, then let go of the encoder

This is `_encode_prompt` from `backend/kaggle_worker.py`, unchanged in shape.
UMT5-XXL is 11.4 GB; on a 15 GB card it would leave no room for the thing that
actually makes the video. It runs in bfloat16 on the CPU, where bf16 is legal,
and the embeddings are cast to fp16 and kept while the encoder is freed.

The tied-embedding repair and the two-probe self-test are carried over from the
worker verbatim. They caught a real failure once - an untied `embed_tokens`
silently producing near-identical embeddings for unrelated prompts - and the
same encoder is in play here.

**This cell downloads ~11.4 GB.** It is the first real spend in this notebook.

In [ ]:
import gc, torch
from diffusers import WanImageToVideoPipeline

NEGATIVE_PROMPT = (
    'blurry, low quality, distorted anatomy, duplicate subjects, static frame, '
    'watermark, text, logo'
)
PROMPT = 'The scene comes to life with gentle natural motion, soft daylight, calm atmosphere'

def free():
    gc.collect()
    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except Exception:
        pass

pipe = None
try:
    free()
    pipe = WanImageToVideoPipeline.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, transformer=None, torch_dtype=torch.bfloat16,
    )
    encoder = pipe.text_encoder
    if (
        encoder is not None
        and hasattr(encoder, 'shared')
        and hasattr(encoder, 'encoder')
        and encoder.encoder.embed_tokens.weight.data_ptr() != encoder.shared.weight.data_ptr()
    ):
        encoder.encoder.embed_tokens.weight = encoder.shared.weight
        print('repaired untied embed_tokens')

    probe_a, _ = pipe.encode_prompt(
        prompt='a red cube on a table', negative_prompt='',
        do_classifier_free_guidance=True, device=torch.device('cpu'), dtype=torch.bfloat16,
    )
    probe_b, _ = pipe.encode_prompt(
        prompt='an astronaut riding a horse on mars', negative_prompt='',
        do_classifier_free_guidance=True, device=torch.device('cpu'), dtype=torch.bfloat16,
    )
    a, b = probe_a.float()[0], probe_b.float()[0]
    live = (a.abs().sum(-1) > 1e-6) | (b.abs().sum(-1) > 1e-6)
    cosine = torch.nn.functional.cosine_similarity(a[live].flatten(), b[live].flatten(), dim=0).item()
    print(f'self-test: {int(live.sum())} live tokens, cosine {cosine:.4f} (want < 0.999)')
    assert int(live.sum()) and cosine <= 0.999, 'UMT5 self-test failed - embeddings are degenerate.'

    positive, negative = pipe.encode_prompt(
        prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT,
        do_classifier_free_guidance=True, device=torch.device('cpu'), dtype=torch.bfloat16,
    )
    positive = positive.to('cpu', torch.float16)
    negative = negative.to('cpu', torch.float16)
    print('embeddings:', tuple(positive.shape), positive.dtype)
finally:
    del pipe
    free()

print('text encoder released.')

## Cell 6 - phase 2: generate, with the card watched while it happens

Transformer in **fp16** on the GPU, VAE in fp32, and `enable_model_cpu_offload()`
so the two are never resident together. The VAE is called before the loop (to
encode the reference frame) and after it (to decode), and the offload hooks
handle that ordering on their own.

VRAM is read two ways, because EXP-006 got this wrong in both directions:
`torch.cuda.max_memory_allocated()` sees only the allocator, and an
`nvidia-smi` call after the run sees an idle card. A background thread samples
the card **during** the run; the allocator number is printed beside it.

If this raises out-of-memory, the ladder is printed rather than retried
automatically - each rung costs real GPU minutes and the choice is yours.

In [ ]:
import threading, time, subprocess
import numpy as np
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

peak = {'mib': 0}
stop = threading.Event()

def sample():
    while not stop.is_set():
        try:
            out = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=5,
            ).stdout
            peak['mib'] = max(peak['mib'], max(int(v) for v in out.split()))
        except Exception:
            pass
        time.sleep(1)

watcher = threading.Thread(target=sample, daemon=True)
watcher.start()

pipe = vae = frames = None
started = time.time()
try:
    torch.cuda.reset_peak_memory_stats()
    vae = AutoencoderKLWan.from_pretrained(
        MODEL_ID, subfolder='vae', revision=MODEL_REVISION, torch_dtype=torch.float32,
    )
    pipe = WanImageToVideoPipeline.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, vae=vae,
        text_encoder=None, tokenizer=None, torch_dtype=torch.float16,
    )
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=5.0)
    pipe.enable_model_cpu_offload()      # do NOT also call pipe.to('cuda')
    try:
        pipe.vae.enable_tiling()
    except Exception as exc:
        print('vae tiling unavailable:', exc)

    generator = torch.Generator(device='cuda').manual_seed(1234)

    def on_step_end(_pipe, step, _timestep, kwargs):
        print(f'  step {step + 1}/{STEPS}  {time.time() - started:7.1f}s', flush=True)
        return kwargs

    frames = pipe(
        image=image,
        prompt=None,
        negative_prompt=None,
        prompt_embeds=positive.to('cuda'),
        negative_prompt_embeds=negative.to('cuda'),
        height=HEIGHT, width=WIDTH, num_frames=FRAMES,
        num_inference_steps=STEPS, guidance_scale=GUIDANCE,
        generator=generator,
        callback_on_step_end=on_step_end,
    ).frames[0]

except torch.cuda.OutOfMemoryError as exc:
    print('OUT OF MEMORY:', exc)
    print()
    print('Retry ladder - one rung per run, cheapest first:')
    print('  1. FRAMES 97 -> 65   (4.0 s -> 2.7 s at 24 fps)')
    print('  2. WIDTH/HEIGHT 832x480 -> 640x368')
    print('  3. swap enable_model_cpu_offload() for enable_sequential_cpu_offload()')
    print('     (much slower per step - if this is the rung that works, check criterion 3)')
    print('If rung 3 still OOMs, KILL CRITERION 2 IS MET. Record it and stop.')
    raise
finally:
    stop.set()
    watcher.join(timeout=3)

elapsed = time.time() - started
allocator_gb = torch.cuda.max_memory_allocated() / 1e9

arr = np.asarray(frames[0], dtype=np.float32)
black = float(arr.mean()) < 1.0
nan = not np.isfinite(np.asarray(frames[len(frames) // 2], dtype=np.float32)).all()

OUTPUT = Path('/kaggle/working/exp007.mp4')
export_to_video(frames, str(OUTPUT), fps=FPS)

print()
print(f'wall clock      : {elapsed:.0f} s   ({elapsed / 60:.1f} min)   [criterion 3: <= 1500 s]')
print(f'peak nvidia-smi : {peak["mib"] / 1024:.2f} GB          [criterion 2: <= ~14 GB]')
print(f'peak allocator  : {allocator_gb:.2f} GB')
print(f'frame 0 mean    : {arr.mean():.1f}  -> black: {black}   [criterion 5]')
print(f'mid-frame NaN   : {nan}                        [criterion 5]')
print(f'output          : {OUTPUT.stat().st_size:,} B')

del pipe, vae
free()

## Cell 7 - criterion 4: did it actually look at the image?

The reference on the left, the first generated frame on the right. This is a
judgement call and it is the one that decides whether any of the rest matters:
if frame 0 does not visibly descend from the picture, this is text-to-video
with a 34 GB download attached.

Watch the clip for the second half of the same question - whether the motion
grows out of the still or drifts away from it.

In [ ]:
import matplotlib.pyplot as plt
import IPython.display as ipd
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].imshow(Image.open(REFERENCE).convert('RGB').resize((WIDTH, HEIGHT)))
axes[0].set_title('reference (uploaded)')
axes[1].imshow(frames[0])
axes[1].set_title('frame 0 (generated)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'{OUTPUT.stat().st_size:>12,} B  {OUTPUT.name}')
print('a real Wan2.1 worker clip today is about 405,459 B')
ipd.display(ipd.Video(str(OUTPUT), embed=True, width=520))

## Cell 8 - record the verdict

Fill this in and paste it into `README.md`. A negative result recorded is worth
more than an experiment quietly abandoned - and on this one, a negative result
also settles what the reference image is for in the shipped product.

In [ ]:
verdict = {
    'date': '2026-09-__',
    'diffusers_version': '',      # cell 3
    'pipeline_available': None,   # cell 3, criterion 1
    'free_disk_gb': None,         # cell 1, criterion 1
    'seconds_per_clip': None,     # cell 6, criterion 3
    'peak_vram_gb': None,         # cell 6 nvidia-smi, criterion 2
    'resolution': '832x480',      # after any rung of the retry ladder
    'frames': None,
    'follows_reference_image': None,  # cell 7, criterion 4 - your own eyes
    'fp16_stable': None,          # cell 6 black/NaN, criterion 5
    'output_bytes': None,
    'decision': '',               # 'proceed to phase 3' | 'rejected: <reason>'
}
for k, v in verdict.items():
    print(f'{k:24}: {v}')